In [17]:

import os

PROJECT = "/content/rag_ollama_chatbot"

os.makedirs(PROJECT, exist_ok=True)

files = [
    "setup.py",
    "config.py",
    "document_processor.py",
    "embeddings.py",
    "vector_store.py",
    "llm.py",
    "app.py"
]

for file in files:
    path = os.path.join(PROJECT, file)
    if not os.path.exists(path):
        open(path, "w").close()

print("Project structure created!")


Project structure created!


In [18]:

%%writefile /content/rag_ollama_chatbot/config.py

OLLAMA_URL = "http://localhost:11434"

LLM_MODEL = "llama3.2:1b"

EMBEDDING_MODEL = "nomic-embed-text"

CHUNK_SIZE = 800

CHUNK_OVERLAP = 150

COLLECTION_NAME = "pdf_documents"


Overwriting /content/rag_ollama_chatbot/config.py


In [19]:

%%writefile /content/rag_ollama_chatbot/setup.py

import subprocess
import sys
import time
import requests

def install_dependencies():

    subprocess.run(
        ["bash", "-lc",
         "apt-get update -qq && apt-get install -y -qq zstd curl"],
        check=True
    )

    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True
    )

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "pypdf", "chromadb", "gradio", "requests"
    ], check=True)

def start_ollama():

    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    for _ in range(30):
        try:
            requests.get(
                "http://localhost:11434/api/tags",
                timeout=2
            )
            break
        except requests.exceptions.RequestException:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama failed to start")

    return process

def download_models():

    subprocess.run(
        ["ollama", "pull", "llama3.2:1b"],
        check=True
    )

    subprocess.run(
        ["ollama", "pull", "nomic-embed-text"],
        check=True
    )

if __name__ == "__main__":

    install_dependencies()
    start_ollama()
    download_models()

    print("Setup completed!")


Overwriting /content/rag_ollama_chatbot/setup.py


In [20]:

%%writefile /content/rag_ollama_chatbot/document_processor.py

from pypdf import PdfReader
from config import CHUNK_SIZE, CHUNK_OVERLAP

def extract_text(pdf_path):

    reader = PdfReader(pdf_path)

    text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    if not text.strip():
        raise ValueError("No readable text found in PDF")

    return text

def split_text(text):

    chunks = []

    start = 0

    while start < len(text):

        end = start + CHUNK_SIZE

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks


Overwriting /content/rag_ollama_chatbot/document_processor.py


In [21]:

%%writefile /content/rag_ollama_chatbot/embeddings.py

import requests
from config import OLLAMA_URL, EMBEDDING_MODEL

def generate_embeddings(texts):

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": EMBEDDING_MODEL,
            "input": texts
        },
        timeout=300
    )

    response.raise_for_status()

    return response.json()["embeddings"]


Overwriting /content/rag_ollama_chatbot/embeddings.py


In [22]:

%%writefile /content/rag_ollama_chatbot/vector_store.py

import chromadb

from config import COLLECTION_NAME
from embeddings import generate_embeddings

client = chromadb.Client()

collection = None

def store_documents(chunks):

    global collection

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        COLLECTION_NAME
    )

    embeddings = generate_embeddings(chunks)

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=embeddings
    )

    return len(chunks)

def retrieve_documents(question, top_k=4):

    if collection is None:
        raise ValueError("Please upload a PDF first")

    embedding = generate_embeddings([question])[0]

    results = collection.query(
        query_embeddings=[embedding],
        n_results=min(top_k, collection.count())
    )

    return results["documents"][0]


Overwriting /content/rag_ollama_chatbot/vector_store.py


In [23]:
%%writefile /content/rag_ollama_chatbot/llm.py

import requests
from config import OLLAMA_URL, LLM_MODEL

def query_llm(prompt):
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": LLM_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        return f"Error communicating with LLM: {str(e)}"

def answer_question(context_chunks, question):
    context = "\n\n".join(context_chunks)
    prompt = f"""Use the following context to answer the user's question.
If you do not know the answer based on the context, say that you do not know.

Context:
{context}

Question: {question}

Answer:"""
    return query_llm(prompt)

Overwriting /content/rag_ollama_chatbot/llm.py


In [24]:
%%writefile /content/rag_ollama_chatbot/app.py

import sys
import os

# Ensure the project directory is in the import path
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

import gradio as gr
from document_processor import extract_text, split_text
from vector_store import store_documents, retrieve_documents
from llm import answer_question

def process_pdf(file):
    if file is None:
        return "Please upload a PDF file."
    try:
        text = extract_text(file.name)
        chunks = split_text(text)
        num_chunks = store_documents(chunks)
        return f"Successfully processed PDF and created {num_chunks} chunks for search!"
    except Exception as e:
        return f"Error processing PDF: {str(e)}"

def chat(question, history):
    try:
        relevant_chunks = retrieve_documents(question)
        answer = answer_question(relevant_chunks, question)
        return answer
    except Exception as e:
        return f"Error generating response: {str(e)}"

with gr.Blocks() as demo:
    gr.Markdown("# Local RAG Chatbot with Ollama")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            upload_button = gr.Button("Process PDF")
            status_output = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            chatbot = gr.ChatInterface(
                fn=chat,
                title="Chat with your PDF"
            )

    upload_button.click(
        fn=process_pdf,
        inputs=[pdf_input],
        outputs=[status_output]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Overwriting /content/rag_ollama_chatbot/app.py


In [25]:
# Run the setup script to install dependencies, download Ollama, and fetch the models
!python /content/rag_ollama_chatbot/setup.py

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ERROR: pip's dependency resolver does not currently take into ac

In [ ]:
# Run the Gradio interface application
!python /content/rag_ollama_chatbot/app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://cf5c94047649f9ad7e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
